## Setup


In [1]:
import torch
from PIL import Image
from torch import Tensor
from torchvision.transforms import ToTensor
from tqdm import tqdm
import matplotlib.pyplot as plt
from torchmetrics.image import PeakSignalNoiseRatio
import pickle
import zlib
import torch.nn as nn
from torchsummary import summary
import torch.optim as optim
from torchvision import models
import pytorch_msssim
import os
import zipfile
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np
import torch.nn.functional as F
from torchvision.models import vgg16, VGG16_Weights

from NeuralCompression.neuralcompression import HyperpriorCompressedOutput, HyperpriorOutput
import NeuralCompression.neuralcompression.functional as ncF

device = "cuda" if torch.cuda.is_available() else "cpu"

/home/iot/Desktop/IoT/NeuralCompression/neuralcompression/__init__.py:21: UserWarning: Could not retrieve neuralcompression version!
  warnings.warn("Could not retrieve neuralcompression version!")


In [2]:
def pickle_size_of(obj, compression_level=6):
    pickled_data = pickle.dumps(obj)
    compressed_data = zlib.compress(pickled_data, level=compression_level)
    return len(compressed_data)

In [3]:
model = torch.hub.load("facebookresearch/NeuralCompression", "msillm_quality_vlo1")
#model = model.to(device)
model = model.eval()
model.update()
model.update_tensor_devices("compress")

Using cache found in /home/iot/.cache/torch/hub/facebookresearch_NeuralCompression_main
/home/iot/.cache/torch/hub/facebookresearch_NeuralCompression_main/neuralcompression/__init__.py:21: UserWarning: Could not retrieve neuralcompression version!
  warnings.warn("Could not retrieve neuralcompression version!")


## Student Model

In [4]:
teacher = model.encoder
teacher.to(device)

HiFiCEncoder(
  (blocks): Sequential(
    (0): Sequential(
      (0): Conv2d(3, 60, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3))
      (1): ChannelNorm2D()
      (2): ReLU()
    )
    (1): Sequential(
      (0): Conv2d(60, 120, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): ChannelNorm2D()
      (2): ReLU()
    )
    (2): Sequential(
      (0): Conv2d(120, 240, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): ChannelNorm2D()
      (2): ReLU()
    )
    (3): Sequential(
      (0): Conv2d(240, 480, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): ChannelNorm2D()
      (2): ReLU()
    )
    (4): Sequential(
      (0): Conv2d(480, 960, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): ChannelNorm2D()
      (2): ReLU()
    )
    (5): Conv2d(960, 220, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
)

In [5]:
class ChannelNorm2D(nn.Module):
    def __init__(self, input_channels: int, epsilon: float = 1e-3, affine: bool = True):
        super().__init__()

        if input_channels <= 1:
            raise ValueError(
                "ChannelNorm only valid for channel counts greater than 1."
            )

        self.epsilon = epsilon
        self.affine = affine

        if affine is True:
            self.gamma = nn.Parameter(torch.ones(1, input_channels, 1, 1))
            self.beta = nn.Parameter(torch.zeros(1, input_channels, 1, 1))

    def forward(self, x: Tensor) -> Tensor:
        mean = torch.mean(x, dim=1, keepdim=True)
        variance = torch.var(x, dim=1, keepdim=True)

        x_normed = (x - mean) * torch.rsqrt(variance + self.epsilon)

        if self.affine is True:
            x_normed = self.gamma * x_normed + self.beta

        return x_normed

In [6]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size=kernel_size,
                                  stride=stride, padding=padding, groups=in_channels)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

class StudentEncoderBase(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.Sequential(
            nn.Sequential(
                DepthwiseSeparableConv(3, 60, kernel_size=7, stride=1, padding=3),
                ChannelNorm2D(60),
                nn.ReLU()
            ),
            nn.Sequential(
                DepthwiseSeparableConv(60, 120, kernel_size=3, stride=2, padding=1),
                ChannelNorm2D(120),
                nn.ReLU()
            ),
            nn.Sequential(
                DepthwiseSeparableConv(120, 240, kernel_size=3, stride=2, padding=1),
                ChannelNorm2D(240),
                nn.ReLU()
            ),
            nn.Sequential(
                DepthwiseSeparableConv(240, 480, kernel_size=3, stride=2, padding=1),
                ChannelNorm2D(480),
                nn.ReLU()
            ),
            nn.Sequential(
                DepthwiseSeparableConv(480, 960, kernel_size=3, stride=2, padding=1),
                ChannelNorm2D(960),
                nn.ReLU()
            ),
            DepthwiseSeparableConv(960, 220, kernel_size=3, stride=1, padding=1)
        )

    def forward(self, x):
        return self.blocks(x)

In [ ]:
# class StudentEncoderPruned(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.blocks = nn.Sequential(
#             nn.Sequential(
#                 DepthwiseSeparableConv(3, 120, kernel_size=7, stride=1, padding=3),
#                 nn.AvgPool2d(kernel_size=2),
#                 ChannelNorm2D(120),
#                 nn.ReLU()
#             ),

#             nn.Sequential(
#                 DepthwiseSeparableConv(120, 480, kernel_size=3, stride=2, padding=1),
#                 nn.AvgPool2d(kernel_size=2),
#                 ChannelNorm2D(480),
#                 nn.ReLU()
#             ),
#             DepthwiseSeparableConv(480, 220, kernel_size=3, stride=1, padding=1),
#             nn.AvgPool2d(kernel_size=2),
#         )

#     def forward(self, x):
#         return self.blocks(x)

In [7]:
student = StudentEncoderBase()
student.to(device)
summary(student, (3, 224, 224))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1          [-1, 3, 224, 224]             150
            Conv2d-2         [-1, 60, 224, 224]             240
DepthwiseSeparableConv-3         [-1, 60, 224, 224]               0
     ChannelNorm2D-4         [-1, 60, 224, 224]               0
              ReLU-5         [-1, 60, 224, 224]               0
            Conv2d-6         [-1, 60, 112, 112]             600
            Conv2d-7        [-1, 120, 112, 112]           7,320
DepthwiseSeparableConv-8        [-1, 120, 112, 112]               0
     ChannelNorm2D-9        [-1, 120, 112, 112]               0
             ReLU-10        [-1, 120, 112, 112]               0
           Conv2d-11          [-1, 120, 56, 56]           1,200
           Conv2d-12          [-1, 240, 56, 56]          29,040
DepthwiseSeparableConv-13          [-1, 240, 56, 56]               0
    ChannelNorm2D-14      

In [ ]:

#model.to(device)
#summary(model.encoder, (3, 224, 224))

## Loss Functions


MS-SSIM (Structural Similarity)

In [8]:
msssim_loss = lambda x, y: 1 - pytorch_msssim.ms_ssim(x, y, data_range=1.0, size_average=True)

VGG Perceptual Loss (Semantic Similarity)

In [ ]:
class VGGPerceptual(nn.Module):
    def __init__(self, perc_layers=None):
        super().__init__()
        weights = VGG16_Weights.IMAGENET1K_V1
        self.vgg = vgg16(weights=weights).features.eval()
        for p in self.vgg.parameters():
            p.requires_grad = False

        # Default perceptual layers
        self.perc_layers = perc_layers or [3, 8, 15, 22]
        self.loss_fn = nn.MSELoss()

        # ImageNet normalization
        self.register_buffer('mean',  torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def normalize(self, x):
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        return (x - self.mean) / self.std

    def forward(self, x, y):
        x = self.normalize(x)
        y = self.normalize(y)

        loss = 0
        for i, layer in enumerate(self.vgg):
            x = layer(x)
            y = layer(y)
            if i in self.perc_layers:
                loss += self.loss_fn(x, y)
        return loss

Distillation Loss - Cosine for Direction and MSE for Magnitude

In [9]:
class DistillationLoss(nn.Module):
    def __init__(self, alpha=0.5, beta=0.5):
        """
        alpha: weight for cosine similarity loss
        beta: weight for MSE loss
        """
        super(DistillationLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.mse = nn.MSELoss()

    def forward(self, student_feat, teacher_feat):
        # Detach teacher so gradients don't flow through it
        teacher_feat = teacher_feat.detach()

        # Global Average Pooling: [B, C, H, W] -> [B, C]
        s_pooled = F.adaptive_avg_pool2d(student_feat, (1, 1)).squeeze(-1).squeeze(-1)
        t_pooled = F.adaptive_avg_pool2d(teacher_feat, (1, 1)).squeeze(-1).squeeze(-1)

        # Cosine Similarity Loss: want vectors pointing in same direction => 1 - cos_sim
        cosine_loss = 1 - F.cosine_similarity(s_pooled, t_pooled, dim=1).mean()

        # Standard MSE Loss over full feature maps
        mse_loss = self.mse(student_feat, teacher_feat)

        # Combined Hybrid Loss
        loss = self.alpha * cosine_loss + self.beta * mse_loss
        return loss

Hyper Params

In [10]:
# Hyperparameters (regularization weights)
alpha_hint1 = 1.0   # weight for first hint loss
alpha_hint2 = 1.0   # weight for second hint loss
beta_latent = 1.0   # weight for latent loss
gamma_msssim = 0.1 # weight for MS-SSIM loss
gamma_perc = 0.01    # weight for perceptual loss

## Training

In [11]:
# Feature storage for hints
teacher_feats = {}
student_feats = {}

# Hook registration
# Assuming teacher.encoder layers accessible as .conv1, .conv2, etc.
def get_teacher_hook(name):
    def hook(module, input, output):
        teacher_feats[name] = output.detach().cpu()
    return hook

def get_student_hook(name):
    def hook(module, input, output):
        student_feats[name] = output.detach().cpu()
    return hook

# Register hooks on desired hint layers
teacher.blocks[1].register_forward_hook(get_teacher_hook('hint1'))
teacher.blocks[3].register_forward_hook(get_teacher_hook('hint2'))

student.blocks[1].register_forward_hook(get_student_hook('hint1'))
student.blocks[3].register_forward_hook(get_student_hook('hint2'))

In [12]:
optimizer = optim.Adam(student.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.9)


distillation_loss = DistillationLoss()
#perceptual_loss = VGGPerceptual()
#perceptual_loss.to(device)

Freeze the Teacher

In [14]:
for p in model.parameters():
    p.requires_grad = False

model.decoder.to("cpu")

HiFiCGenerator(
  (block_0): Sequential(
    (0): ChannelNorm2D()
    (1): Conv2d(220, 960, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (2): ChannelNorm2D()
  )
  (resid_blocks): Sequential(
    (0): _ResidualBlock(
      (sequence): Sequential(
        (0): Conv2d(960, 960, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ChannelNorm2D()
        (2): ReLU()
        (3): Conv2d(960, 960, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): ChannelNorm2D()
      )
    )
    (1): _ResidualBlock(
      (sequence): Sequential(
        (0): Conv2d(960, 960, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ChannelNorm2D()
        (2): ReLU()
        (3): Conv2d(960, 960, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): ChannelNorm2D()
      )
    )
    (2): _ResidualBlock(
      (sequence): Sequential(
        (0): Conv2d(960, 960, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ChannelNorm2D()
        (2): Re

In [15]:
from tqdm import tqdm
import matplotlib.pyplot as plt

def train_epoch(dataloader, epoch=None):
    student.train()
    teacher.eval()
    running_loss = 0.0
    total_hint1_loss = 0.0
    total_hint2_loss = 0.0
    total_latent_loss = 0.0
    total_ssim_loss = 0.0
    total_perc_loss = 0.0

    # Add TQDM loader
    loop = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch if epoch is not None else ''}")

    for i, x in loop:

        # Padding Correction        
        x = x.to(device)
        x, (height, width) = ncF.pad_image_to_factor(x, model._factor)

        # Latent Generation
        with torch.no_grad():
            t_latent = teacher(x)
        s_latent = student(x)

        optimizer.zero_grad()

        # Hint losses (logit matching)
        hint1_loss = distillation_loss(student_feats['hint1'].to(device), teacher_feats['hint1'].to(device))
        hint2_loss = distillation_loss(student_feats['hint2'].to(device), teacher_feats['hint2'].to(device))
        latent_loss = distillation_loss(s_latent, t_latent)

        # Reconstruction via Teacher Decoder
        #x_recon = model.decoder(s_latent).to(device)

        # Recon Loss
        # perc_loss = perceptual_loss(x, x_recon)
        # ssim_loss = msssim_loss(x, x_recon)

        # Cumulative Loss
        loss = (alpha_hint1 * hint1_loss
                + alpha_hint2 * hint2_loss
                + beta_latent * latent_loss)
       #         + gamma_msssim * ssim_loss
        #        + gamma_perc * perc_loss)

        # Backprop and optimize
        loss.backward()
        optimizer.step()

        # Accumulate loss values
        running_loss += loss.item() * x.size(0)
        total_hint1_loss += hint1_loss.item() * x.size(0)
        total_hint2_loss += hint2_loss.item() * x.size(0)
        total_latent_loss += latent_loss.item() * x.size(0)
        # total_ssim_loss += ssim_loss.item() * x.size(0)
        # total_perc_loss += perc_loss.item() * x.size(0)

    # Average losses
    dataset_size = len(dataloader.dataset)
    epoch_loss = running_loss / dataset_size
    avg_hint1_loss = total_hint1_loss / dataset_size
    avg_hint2_loss = total_hint2_loss / dataset_size
    avg_latent_loss = total_latent_loss / dataset_size
    # avg_ssim_loss = total_ssim_loss / dataset_size
    # avg_perc_loss = total_perc_loss / dataset_size

    print(f"\n[Epoch {epoch}] Component-wise Losses:")
    print(f"Hint1 Loss:     {avg_hint1_loss:.4f}")
    print(f"Hint2 Loss:     {avg_hint2_loss:.4f}")
    print(f"Latent Loss:    {avg_latent_loss:.4f}")
    # print(f"MS-SSIM Loss:   {avg_ssim_loss:.4f}")
    # print(f"Perceptual Loss:{avg_perc_loss:.4f}")
    print(f"Total Loss:     {epoch_loss:.4f}")

    # # Plot reconstructed image after the epoch
    # x_vis = x[0].detach().cpu().permute(1, 2, 0)
    # #x_recon_vis = x_recon[0].detach().cpu().permute(1, 2, 0)

    # fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    # axs[0].imshow(x_vis)
    # axs[0].set_title("Original Image")
    # axs[0].axis('off')

    # axs[1].imshow(x_recon_vis)
    # axs[1].set_title("Reconstructed Image")
    # axs[1].axis('off')

    # plt.suptitle(f"Reconstruction at Epoch {epoch}")
    # plt.show()

    return epoch_loss


## Dataset

In [16]:
path = "/home/iot/Desktop/IoT/datasets/CLIC/archive/val2017"

In [17]:
# 2. Create custom Dataset class
class CLICDataset(Dataset):
    def __init__(self, root_dir, transform=None, split='train'):
        self.root_dir = os.path.join(root_dir, split)
        self.transform = transform
        self.image_files = [f for f in os.listdir(self.root_dir)
                           if f.endswith(('.png', '.jpg', '.jpeg'))]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_files[idx])
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image

# 3. Set up transformations # LOOKUP the original PAPER's transformations
transform = transforms.Compose([
    transforms.ToTensor(),
])



# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the MIT license found in the
# LICENSE file in the root directory of this source tree.

from torchvision.transforms import (
    CenterCrop,
    Compose,
    RandomChoice,
    RandomCrop,
    RandomHorizontalFlip,
    RandomResizedCrop,
    Resize,
    ToTensor,
)


def default_train_transform(image_size: int) -> Compose:
    choice_transform = RandomChoice(
        [
            RandomCrop(size=image_size, pad_if_needed=True, padding_mode="reflect"),
            RandomResizedCrop(size=image_size),
        ]
    )
    return Compose(
        [
            choice_transform,
            RandomHorizontalFlip(),
            ToTensor(),
        ]
    )

transform = default_train_transform(224)

# 4. Create datasets and dataloaders
batch_size = 32

train_dataset = CLICDataset(root_dir= os.path.join(path),
                           transform=transform,
                           split='train')

train_loader = DataLoader(train_dataset,
                         batch_size=batch_size,
                         shuffle=True,)

# Test the dataloader
batch = next(iter(train_loader))
print(f"Batch shape: {batch.shape}")  # Should be [batch_size, 3, 256, 256]
print(f"No of Batches = {len(train_loader)}")

Batch shape: torch.Size([32, 3, 224, 224])
No of Batches = 157


In [18]:
# Example training loop
torch.cuda.empty_cache()  # Frees cached memory (not allocated memory)
num_epochs = 10

for epoch in range(num_epochs):
    train_loss = train_epoch(train_loader)
    ##scheduler.step()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {train_loss:.4f}")

Epoch :   0%|          | 0/157 [00:00<?, ?it/s]

Epoch :   0%|          | 0/157 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 480.00 MiB. GPU 0 has a total capacity of 23.53 GiB of which 274.06 MiB is free. Process 24733 has 692.00 MiB memory in use. Process 25454 has 21.07 GiB memory in use. Including non-PyTorch memory, this process has 1018.00 MiB memory in use. Of the allocated memory 551.79 MiB is allocated by PyTorch, and 16.21 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
torch.save(student.state_dict(), "/home/iot/Desktop/IoT/models/student800_100epochs-1e3")

## Quantization Aware Training

In [ ]:
# Custom QAT configuration (adjust as needed)
qat_config = QConfig(
    activation=default_fake_quant.with_args(observer=torch.ao.quantization.MovingAverageMinMaxObserver,
                                           quant_min=0,
                                           quant_max=255,
                                           dtype=torch.quint8),
    weight=default_weight_fake_quant.with_args(observer=torch.ao.quantization.MinMaxObserver,
                                              quant_min=-128,
                                              quant_max=127,
                                              dtype=torch.qint8)
)

# Apply configuration to student model
student.qconfig = qat_config

In [ ]:
# Prepare model for QAT (inserts fake quantization modules)
student_prepared = prepare_qat(student, inplace=False).to(device)
student = student_prepared
# If using FX Graph Mode (recommended for complex models):
# qconfig_mapping = get_default_qat_qconfig_mapping()
# student_prepared = prepare_fx(student, qconfig_mapping, example_inputs=torch.randn(1,3,224,224).to(device)

In [ ]:
# Set to evaluation mode and convert
student_prepared.eval()
student_quantized = convert(student_prepared, inplace=False)

# For FX Graph Mode:
# student_quantized = convert_fx(student_prepared)

# Verify quantization
print(student_quantized)